In [2]:
import numpy as np

def knn_vectorized(P: np.ndarray, Q: np.ndarray, k: int):
    """
    P: (N, D) reference points
    Q: (M, D) query points
    Returns: indices (M, k), distances (M, k)
    """
    # ||p||^2 shape (N,), ||q||^2 shape (M,)
    P_sq = np.einsum('nd,nd->n', P, P)  # (N,)
    Q_sq = np.einsum('md,md->m', Q, Q)  # (M,)

    # Cross term Q @ P^T  →  (M, N)
    cross = Q @ P.T                      # (M, N)

    # Broadcast: (M,1) + (1,N) - 2*(M,N)  →  (M, N)
    D_sq = Q_sq[:, None] + P_sq[None, :] - 2 * cross
    D_sq = np.clip(D_sq, 0, None)        # guard floating-point negatives

    # Partial sort — only need top k, not full sort
    idx = np.argpartition(D_sq, k, axis=1)[:, :k]

    # Sort those k by actual distance
    row_idx = np.arange(Q.shape[0])[:, None]
    k_dists = D_sq[row_idx, idx]
    order   = np.argsort(k_dists, axis=1)
    idx     = idx[row_idx, order]
    dists   = np.sqrt(k_dists[row_idx, order])

    return idx, dists


# ── Hard Mode: tiled version for large datasets ──────────────────────────────
def knn_tiled(P: np.ndarray, Q: np.ndarray, k: int, tile_size: int = 512):
    """
    Memory-safe tiled KNN.
    Instead of one (M×N) matrix we process Q in tiles of `tile_size` rows,
    keeping memory at O(tile_size × N) instead of O(M × N).
    """
    M = Q.shape[0]
    all_idx   = np.empty((M, k), dtype=np.int64)
    all_dists = np.empty((M, k), dtype=np.float64)

    P_sq = np.einsum('nd,nd->n', P, P)  # (N,) — computed once

    for start in range(0, M, tile_size):
        end   = min(start + tile_size, M)
        Q_tile = Q[start:end]                        # (t, D)
        Q_sq   = np.einsum('td,td->t', Q_tile, Q_tile)  # (t,)
        cross  = Q_tile @ P.T                            # (t, N)
        D_sq   = Q_sq[:, None] + P_sq[None, :] - 2 * cross
        D_sq   = np.clip(D_sq, 0, None)

        idx   = np.argpartition(D_sq, k, axis=1)[:, :k]
        row_i = np.arange(end - start)[:, None]
        k_d   = D_sq[row_i, idx]
        order = np.argsort(k_d, axis=1)
        idx   = idx[row_i, order]

        all_idx[start:end]   = idx
        all_dists[start:end] = np.sqrt(k_d[row_i, order])

    return all_idx, all_dists

In [3]:
import numpy as np

class Value:
    def __init__(self, data, _children=(), _op='', label=''):
        self.data     = float(data)
        self.grad     = 0.0          # ∂L/∂self — filled during backward
        self._backward = lambda: None  # default: leaf nodes do nothing
        self._prev    = set(_children)
        self._op      = _op
        self.label    = label

    def __repr__(self):
        return f"Value(data={self.data:.4f}, grad={self.grad:.4f}, label='{self.label}')"

    # ── Forward ops ──────────────────────────────────────────────────────────

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out   = Value(self.data * other.data, (self, other), '*')

        def _backward():
            # d(out)/d(self)  = other.data   ← product rule
            # d(out)/d(other) = self.data
            self.grad  += other.data * out.grad
            other.grad += self.data  * out.grad

        out._backward = _backward
        return out

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out   = Value(self.data + other.data, (self, other), '+')

        def _backward():
            # Gradient of addition passes through unchanged (derivative = 1)
            self.grad  += 1.0 * out.grad
            other.grad += 1.0 * out.grad

        out._backward = _backward
        return out

    def tanh(self):
        t   = np.tanh(self.data)
        out = Value(t, (self,), 'tanh')

        def _backward():
            # d(tanh(x))/dx = 1 - tanh²(x)
            self.grad += (1 - t**2) * out.grad

        out._backward = _backward
        return out

    # ── Backward pass ────────────────────────────────────────────────────────

    def backward(self):
        """
        Topological sort guarantees every node's output gradient
        is fully accumulated before we call its _backward().
        """
        topo, visited = [], set()

        def build_topo(node):
            if node not in visited:
                visited.add(node)
                for child in node._prev:
                    build_topo(child)
                topo.append(node)

        build_topo(self)
        self.grad = 1.0          # ∂z/∂z = 1
        for node in reversed(topo):
            node._backward()


# ── Test: z = tanh(a * b + c) ────────────────────────────────────────────────
a = Value(2.0, label='a')
b = Value(3.0, label='b')
c = Value(1.0, label='c')

ab  = a * b          # a·b  = 6.0
s   = ab + c         # +c   = 7.0
z   = s.tanh()       # tanh(7) ≈ 0.9999834
z.label = 'z'

z.backward()

print(f"z     = {z.data:.6f}")
print(f"dz/da = {a.grad:.6f}")  # (1 - tanh²(7)) * b = tiny * 3
print(f"dz/db = {b.grad:.6f}")  # (1 - tanh²(7)) * a = tiny * 2
print(f"dz/dc = {c.grad:.6f}")  # (1 - tanh²(7)) * 1

z     = 0.999998
dz/da = 0.000010
dz/db = 0.000007
dz/dc = 0.000003


In [4]:
from collections import deque
from typing import List

def sliding_window_maximum(nums: List[int], k: int) -> List[int]:
    """
    O(n) sliding window max via monotonic decreasing deque.
    The deque stores *indices*, front is always the index of the current max.
    """
    dq:     deque = deque()   # indices, front = index of max
    result: list  = []

    for i, val in enumerate(nums):

        # ── Step 1: Evict expired indices ─────────────────────────────────────
        # The front is out of the [i-k+1, i] window → remove it
        if dq and dq[0] <= i - k:
            dq.popleft()

        # ── Step 2: Maintain decreasing invariant ─────────────────────────────
        # Pop from the BACK while the back element is ≤ current value.
        # Those elements can never be the max of any future window because
        # val is both larger AND newer (will leave the window later).
        while dq and nums[dq[-1]] <= val:
            dq.pop()

        dq.append(i)

        # ── Step 3: Emit max when window is full ──────────────────────────────
        if i >= k - 1:
            result.append(nums[dq[0]])   # front is always the current max

    return result


# Quick test
nums = [1, 3, -1, -3, 5, 3, 6, 7]
print(sliding_window_maximum(nums, k=3))
# → [3, 3, 5, 5, 6, 7]

[3, 3, 5, 5, 6, 7]
